# Hybrid probability/binomial IRT simulations

Publication-oriented cleanup of the supplied simulation notebook, accompanying the JART presentation by Takayama and Shimizu (15 September 2026). This notebook contains **simulation and estimation code**, not the MMLU empirical analysis on slides 15 onward.

Run cells from top to bottom in a Python 3 environment with NumPy, pandas, SciPy and Matplotlib. The notebook is self-contained; the companion `hybrid_irt.py` exports the same definitions for scripts. No API key, external dataset or network connection is needed for the analysis. Outputs and execution history were cleared for distribution.

The default execution runs the seed-42 recovery example and an explicit shared-theta example. Large replicate grids are opt-in in the last cells; their original settings are fully retained in `EXPERIMENTS`. Changing a run switch to `True` runs the corresponding complete grid. Save generated results under a directory you choose.

## Simulation specification

| Component | Implemented condition |
|---|---|
| Item-response function | `mu[g,j] = sigmoid(1.702 * a[j] * (theta[g] - b[j]))`; one-dimensional 2PL, positive discrimination |
| True discrimination | Independent `Uniform(0.5, 2.0)` per item |
| True difficulty | Independent `Uniform(-2.0, 2.0)` per item |
| True ability | Independent `Normal(0,1)` per respondent in each group; raw draws are not standardized before data generation |
| Items | 100 in the presentation recovery/sample-size experiments; supplementary item-count grids vary 10 to 500 |
| Binomial channel | `r[g,j] ~ Binomial(R_b, mu[g,j])`, with `R_b=5`; counts 0 through 5, not proportions |
| Probability channel | `q[g,j,t] = clip(mu[g,j] + Normal(0, sigma[g,j]**2), 0, 1)` for `R_l=5` repeats |
| Noise SD | `sigma[g,j] = 0.01 + 0.2*sqrt(mu[g,j]*(1-mu[g,j]))`, ranging from about 0.01 to 0.11 |
| Probability summaries | `p = mean(q, axis=repeats)` and `s2 = var(q, ddof=1)`; these are probability-scale values, not log probabilities |
| Analysis variance | `v = s2/R_l + c_tau/R_l`, `c_tau=0.25`; floor is 0.05 when `R_l=5` (slide 11) |
| Randomness | NumPy `default_rng(seed)`; seed 42 for recovery; seeds 42–51 inclusive for ten-replicate grids |
| Conditional independence | Independent draws across repetitions/cells and between the probability and binomial channels, given the true parameters |
| Identification | During fitting, pooled **included** respondents' theta has mean 0 and population SD 1; a and b are transformed to preserve mu |
| Termination | Relative objective change <1e-6 **and** maximum parameter change <1e-4; at most 1000 outer iterations and 1000 item optimizer iterations |

Clipping produces point mass at 0/1 and can shift the probability observation's expected value away from mu near boundaries. The Gaussian objective is a **working likelihood**; it is not the exact clipped data-generating likelihood. The variance floor and item ridge are preserved from the source.

## Observation groups and a shared theta

| Group | Measurement supplied to fitting | Ability parameter |
|---|---|---|
| Full probability | p and s2 at all available items | One `theta_full_probability` per row |
| Full binomial | Correct counts r at all available items | One `theta_binary` per row |
| Mixed / overlapping channels | Binomial counts at all items; probability observations additionally enabled by mask | **One `theta_mixed` per row, shared by both channels** |

For a mixed cell, `mask=1` means **probability + binomial** when both are finite; `mask=0` means binomial only. The mask NEVER removes a finite binomial observation. In the generator, `P(mask=0)=binary_fraction` independently per mixed cell. This is an expected fraction, not an exact fixed count. `binary_fraction=0` gives both channels at every mixed item; `binary_fraction=1` gives binomial only. Binary_fraction does not denote the overall proportion of binomial respondents.

For separate probability-only and binomial-only groups, respondents are different people/models in the simulation. Do not place the same real respondent in both full groups if a shared theta is intended. Use aligned rows in `p_mix/s2_mix/r_mix/mask_mix` for that purpose. With NaN binomial entries, mixed cells may also be probability-only; tests cover this use.

## Connection to the presentation

| Slides 12–14 | Fitted respondents | Notebook experiment |
|---|---|---|
| 1-A, recovery | 100 binomial | `recovery`, `binary_only` |
| 1-B, recovery | 100 probability | `recovery`, `probability_only` |
| 1-C, recovery | 100 probability + 100 binomial, distinct respondents | `recovery`, `no_mixed` |
| 2-A, sample size | Binomial n varies | `sample_binary` |
| 2-B, sample size | Probability n varies | `sample_probability` |
| 2-C, sample size | Binomial n varies + probability n=50 | `sample_binary_plus_probability50` |
| 2-D, sample size | Probability n varies + binomial n=50 | `sample_probability_plus_binary50` |

Sample sizes: 10,20,...,100,200,300,...,1000 (19 sizes), each with seeds 42–51 (10 datasets). Each sample-size family therefore has 190 fits. In the source, additional groups were generated but omitted from some fits. Those unused **random draws** are retained to preserve the generated data for each seed; the catalog distinguishes generated and fitted sample sizes.

Supplementary studies retain mixed respondents, item-count sweeps, mixed fractions 0.05 to 0.95 in steps of 0.05, and fractions 0.25/0.50/0.75 with **50 replicates, seeds 1–50**. The last source comment said 100 replicates, but its executable setting was 50. These supplementary studies are not claimed to be shown on slides 12–14.

## Recovery metrics and standard errors

`rmse_*` compares estimates with true parameters standardized using **only the respondents included in that fit**. `rmse_*_raw` also records the comparison with the original unstandardized generating values, preserving the raw-truth comparison used by the early export cells. These two metrics need not match; do not silently replace numbers on the presentation. The source's later summary standardized using all generated groups, including groups excluded from fitting; that has been corrected here.

`rmsse_* = sqrt(mean(conditional_SE**2))` across items or respondents within a fit. The source named this `mean_*_se`, although it was not the arithmetic mean. Across replicate seeds we report its mean, sample SD and Monte Carlo SE. A parameter's approximate conditional SE and the Monte Carlo SE of a simulation summary describe different uncertainties. Empty/unfitted groups have NaN metrics. Nonconverged fits remain in the output with flags and convergence rates.

The numerical optimization, SE formulas and generator are retained. Corrections are documented in `README_ja.md`: missing/invalid-input checks, fitted-group-only truth alignment, the wrong mixed-item runner call, misleading comments, and inconsistent/reused export variables. This cleanup does not assert that the presentation's rounded values were produced by this exact saved notebook revision.


## Imports and numerical scale

Only NumPy, pandas, SciPy and Matplotlib are required for the analysis. D = 1.702 is the logistic scaling constant used throughout the original notebook. Matplotlib is imported locally by plotting helpers.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar

D = 1.702


## Numerical utilities and identification

After every outer iteration, pool the abilities of the fitted respondents and set their mean to 0 and population SD (ddof=0) to 1. Transform a and b at the same time to preserve D*a*(theta-b). A mixed respondent appears only once in this standardization, even when both channels contribute. Analysis variance is v = (s2 + c_tau)/R_l.


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40.0, 40.0)))


def logit(p, eps=1e-8):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def rmse(x, y):
    if len(x) == 0:
        return np.nan
    return float(np.sqrt(np.mean((np.asarray(x) - np.asarray(y)) ** 2)))


def corr(x, y):
    if len(x) < 2:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def to_numpy_or_empty(x, n_items=None):
    """
    Convert DataFrame/array/None to numpy array.
    If x is None, return an empty array with shape (0, n_items).
    """
    if x is None:
        if n_items is None:
            return np.empty((0, 0))
        return np.empty((0, n_items))
    if isinstance(x, pd.DataFrame):
        return x.to_numpy(dtype=float)
    return np.asarray(x, dtype=float)


def infer_n_items(*arrays):
    for x in arrays:
        if x is None:
            continue
        if isinstance(x, pd.DataFrame):
            return x.shape[1]
        arr = np.asarray(x)
        if arr.ndim == 2:
            return arr.shape[1]
    raise ValueError("At least one non-empty 2D data matrix is required.")


def make_variance_from_s2(s2, R_l, c_tau=0.25):
    """
    Construct probability-channel analysis variance.

    v = s^2 / R_l + tau^2
    tau^2 = c_tau / R_l

    s2 is the empirical variance of repeated probability measurements.
    """
    if s2.size == 0:
        return s2
    tau2 = c_tau / R_l
    return s2 / R_l + tau2

def standardize_params_mixed(a, b, theta_full_p, theta_bin, theta_mix):
    parts = []
    if len(theta_full_p):
        parts.append(theta_full_p)
    if len(theta_bin):
        parts.append(theta_bin)
    if len(theta_mix):
        parts.append(theta_mix)

    if not parts:
        return a, b, theta_full_p, theta_bin, theta_mix

    theta_all = np.concatenate(parts)
    mean = theta_all.mean()
    sd = theta_all.std()

    if not np.isfinite(sd) or sd < 1e-8:
        sd = 1.0

    a_new = a * sd
    b_new = (b - mean) / sd

    theta_full_p_new = (theta_full_p - mean) / sd
    theta_bin_new = (theta_bin - mean) / sd
    theta_mix_new = (theta_mix - mean) / sd

    return a_new, b_new, theta_full_p_new, theta_bin_new, theta_mix_new


## Initialization

Start a at 1 and obtain b/theta from clipped logit mean scores. Item initialization gives equal weight to each nonempty group mean, as in the source. Mixed rows prefer available probabilities for initialization and fall back to r/R_b. This preference is only for initialization; the fitted objective uses both available channels.


In [ ]:
def initialize_params_mixed(
    p_full,
    v_full,
    r_bin,
    p_mix,
    v_mix,
    r_mix,
    mask_mix,
    R_b,
    init_ab=None,
):
    """
    Initialize item parameters and abilities for:
      1. full probability respondents
      2. full binary respondents
      3. item-wise mixed respondents
    """
    n_items = infer_n_items(p_full, r_bin, p_mix, r_mix, mask_mix)

    n_full_p = p_full.shape[0]
    n_bin = r_bin.shape[0]
    n_mix = p_mix.shape[0]

    if init_ab is None:
        item_means = []

        if n_full_p:
            item_means.append(np.nanmean(p_full, axis=0))

        if n_bin:
            item_means.append(np.nanmean(r_bin / R_b, axis=0))

        if n_mix:
            # mixed observed probability equivalent for initialization
            mix_prob_equiv = np.where((mask_mix == 1) & np.isfinite(p_mix), p_mix, r_mix / R_b)
            item_means.append(np.nanmean(mix_prob_equiv, axis=0))

        p_item = np.nanmean(np.vstack(item_means), axis=0)
        p_item = np.clip(p_item, 1e-4, 1.0 - 1e-4)

        a = np.ones(n_items)
        b = -logit(p_item) / D
    else:
        a, b = [np.asarray(z, dtype=float).copy() for z in init_ab]

    # theta initialization
    if n_full_p:
        row_mean = np.nanmean(p_full, axis=1)
        theta_full_p = logit(np.clip(row_mean, 1e-4, 1.0 - 1e-4)) / D
    else:
        theta_full_p = np.array([])

    if n_bin:
        row_mean = np.nanmean(r_bin / R_b, axis=1)
        theta_bin = logit(np.clip(row_mean, 1e-4, 1.0 - 1e-4)) / D
    else:
        theta_bin = np.array([])

    if n_mix:
        mix_prob_equiv = np.full_like(p_mix, np.nan, dtype=float)
        
        prob_available = (
            (mask_mix == 1)
            & np.isfinite(p_mix)
        )
        mix_prob_equiv[prob_available] = p_mix[prob_available]


        bin_available = (
            ~np.isfinite(mix_prob_equiv)
            & np.isfinite(r_mix)
        )
        mix_prob_equiv[bin_available] = r_mix[bin_available] / R_b
        
        row_mean = np.nanmean(mix_prob_equiv, axis=1)
        theta_mix = logit(np.clip(row_mean, 1e-4, 1.0 - 1e-4)) / D
    else:
        theta_mix = np.array([])

    return standardize_params_mixed(a, b, theta_full_p, theta_bin, theta_mix)


## Ability updates: one theta may use both channels

Each theta update minimizes its conditional objective with fixed items. For mixed rows, `mask == 1` enables the probability contribution; EVERY finite binomial count contributes, including at mask-1 items. Thus mask-1 items with finite p, s2 and r use probability + binomial jointly for ONE theta. mask-0 items use binomial only. Bounds: [-6, 6], bounded scalar optimization, xatol=1e-5.


In [ ]:
def probability_theta_objective(theta, p_g, v_g, a, b):
    mu = sigmoid(D * a * (theta - b))
    valid = np.isfinite(p_g) & np.isfinite(v_g)
    if not np.any(valid):
        return 0.0
    return float(0.5 * np.sum((p_g[valid] - mu[valid]) ** 2 / v_g[valid]))


def update_theta_full_probability(p_full, v_full, a, b, bounds=(-6.0, 6.0)):
    if p_full.shape[0] == 0:
        return np.array([])

    theta_hat = np.zeros(p_full.shape[0])

    for g in range(p_full.shape[0]):
        res = minimize_scalar(
            lambda th: probability_theta_objective(th, p_full[g], v_full[g], a, b),
            bounds=bounds,
            method="bounded",
            options={"xatol": 1e-5},
        )
        theta_hat[g] = res.x

    return theta_hat

def binary_theta_objective(theta, r_g, R_b, a, b):
    mu = sigmoid(D * a * (theta - b))
    mu = np.clip(mu, 1e-12, 1.0 - 1e-12)
    valid = np.isfinite(r_g)
    if not np.any(valid):
        return 0.0

    return float(
        -np.sum(
            r_g[valid] * np.log(mu[valid])
            + (R_b - r_g[valid]) * np.log(1.0 - mu[valid])
        )
    )


def update_theta_binary(r_bin, R_b, a, b, bounds=(-6.0, 6.0)):
    if r_bin.shape[0] == 0:
        return np.array([])

    theta_hat = np.zeros(r_bin.shape[0])

    for g in range(r_bin.shape[0]):
        res = minimize_scalar(
            lambda th: binary_theta_objective(th, r_bin[g], R_b, a, b),
            bounds=bounds,
            method="bounded",
            options={"xatol": 1e-5},
        )
        theta_hat[g] = res.x

    return theta_hat

def mixed_theta_objective(theta, p_g, v_g, r_g, mask_g, R_b, a, b):
    """
    mask_g[i] == 1:
        logprobs/probability observation is available.
        Use probability term if p_g and v_g are finite.
        Also use binary term if r_g is finite.

    mask_g[i] == 0:
        logprobs/probability observation is unavailable.
        Use binary term if r_g is finite.
    """
    mu = sigmoid(D * a * (theta - b))
    mu_clip = np.clip(mu, 1e-12, 1.0 - 1e-12)

    value = 0.0

    # probability term: only when mask == 1 and p/v are available
    prob_idx = (
        (mask_g == 1)
        & np.isfinite(p_g)
        & np.isfinite(v_g)
    )

    # binary term: use whenever r is available
    # this means:
    #   mask == 0 -> binary only
    #   mask == 1 with finite r -> probability + binary
    bin_idx = np.isfinite(r_g)

    if np.any(prob_idx):
        value += 0.5 * np.sum(
            (p_g[prob_idx] - mu[prob_idx]) ** 2 / v_g[prob_idx]
        )

    if np.any(bin_idx):
        value += -np.sum(
            r_g[bin_idx] * np.log(mu_clip[bin_idx])
            + (R_b - r_g[bin_idx]) * np.log(1.0 - mu_clip[bin_idx])
        )

    return float(value)


def update_theta_mixed(p_mix, v_mix, r_mix, mask_mix, R_b, a, b, bounds=(-6.0, 6.0)):
    if p_mix.shape[0] == 0:
        return np.array([])

    theta_hat = np.zeros(p_mix.shape[0])

    for g in range(p_mix.shape[0]):
        res = minimize_scalar(
            lambda th: mixed_theta_objective(
                th,
                p_mix[g],
                v_mix[g],
                r_mix[g],
                mask_mix[g],
                R_b,
                a,
                b,
            ),
            bounds=bounds,
            method="bounded",
            options={"xatol": 1e-5},
        )
        theta_hat[g] = res.x

    return theta_hat


## Item updates and objective monitoring

Optimize (log(a), b) for each item with an analytic gradient and L-BFGS-B. Both mixed contributions use the same theta_mixed. Conditional bounds are a in [0.05, 5], b in [-5, 5]; subsequent scale identification can move the reported values outside those bounds. The item penalty is ridge*((log(a))**2 + b**2), ridge=1e-4. Constants independent of parameters are omitted from the objective. Item optimizer: ftol=1e-7, gtol=1e-6, maxls=30.


In [ ]:
def item_objective_and_grad_mixed(
    z,
    p_full_j,
    v_full_j,
    r_bin_j,
    p_mix_j,
    v_mix_j,
    r_mix_j,
    mask_mix_j,
    R_b,
    theta_full_p,
    theta_bin,
    theta_mix,
    ridge=1e-4,
):
    alpha, bj = z
    aj = np.exp(alpha)

    value = 0.0
    grad_alpha = 0.0
    grad_b = 0.0

    # 1. full probability group
    if len(theta_full_p):
        valid = np.isfinite(p_full_j) & np.isfinite(v_full_j)
        if np.any(valid):
            th = theta_full_p[valid]
            eta = D * aj * (th - bj)
            mu = sigmoid(eta)
            residual = mu - p_full_j[valid]
            w = 1.0 / v_full_j[valid]

            value += 0.5 * np.sum(w * residual ** 2)

            dL_deta = w * residual * mu * (1.0 - mu)
            d_eta_da = D * (th - bj)
            d_eta_db = -D * aj

            grad_alpha += np.sum(dL_deta * d_eta_da) * aj
            grad_b += np.sum(dL_deta * d_eta_db)

    # 2. full binary group
    if len(theta_bin):
        valid = np.isfinite(r_bin_j)
        if np.any(valid):
            th = theta_bin[valid]
            eta = D * aj * (th - bj)
            mu = np.clip(sigmoid(eta), 1e-12, 1.0 - 1e-12)

            rj = r_bin_j[valid]

            value += -np.sum(
                rj * np.log(mu)
                + (R_b - rj) * np.log(1.0 - mu)
            )

            dL_deta = R_b * mu - rj
            d_eta_da = D * (th - bj)
            d_eta_db = -D * aj

            grad_alpha += np.sum(dL_deta * d_eta_da) * aj
            grad_b += np.sum(dL_deta * d_eta_db)

    # 3. mixed group: probability part
    if len(theta_mix):
        prob_idx = (mask_mix_j == 1) & np.isfinite(p_mix_j) & np.isfinite(v_mix_j)

        if np.any(prob_idx):
            th = theta_mix[prob_idx]
            eta = D * aj * (th - bj)
            mu = sigmoid(eta)
            residual = mu - p_mix_j[prob_idx]
            w = 1.0 / v_mix_j[prob_idx]

            value += 0.5 * np.sum(w * residual ** 2)

            dL_deta = w * residual * mu * (1.0 - mu)
            d_eta_da = D * (th - bj)
            d_eta_db = -D * aj

            grad_alpha += np.sum(dL_deta * d_eta_da) * aj
            grad_b += np.sum(dL_deta * d_eta_db)

        # 4. mixed group: binary part
        bin_idx = np.isfinite(r_mix_j)

        if np.any(bin_idx):
            th = theta_mix[bin_idx]
            eta = D * aj * (th - bj)
            mu = np.clip(sigmoid(eta), 1e-12, 1.0 - 1e-12)

            rj = r_mix_j[bin_idx]

            value += -np.sum(
                rj * np.log(mu)
                + (R_b - rj) * np.log(1.0 - mu)
            )

            dL_deta = R_b * mu - rj
            d_eta_da = D * (th - bj)
            d_eta_db = -D * aj

            grad_alpha += np.sum(dL_deta * d_eta_da) * aj
            grad_b += np.sum(dL_deta * d_eta_db)

    # ridge
    if ridge is not None and ridge > 0:
        value += ridge * (alpha ** 2 + bj ** 2)
        grad_alpha += 2.0 * ridge * alpha
        grad_b += 2.0 * ridge * bj

    return float(value), np.array([grad_alpha, grad_b])

def update_items_mixed(
    p_full,
    v_full,
    r_bin,
    p_mix,
    v_mix,
    r_mix,
    mask_mix,
    R_b,
    a,
    b,
    theta_full_p,
    theta_bin,
    theta_mix,
    item_maxiter=50,
    ridge=1e-4,
):
    n_items = len(a)
    new_a = a.copy()
    new_b = b.copy()

    for j in range(n_items):
        p_full_j = p_full[:, j] if p_full.shape[0] else np.array([])
        v_full_j = v_full[:, j] if v_full.shape[0] else np.array([])
        r_bin_j = r_bin[:, j] if r_bin.shape[0] else np.array([])

        p_mix_j = p_mix[:, j] if p_mix.shape[0] else np.array([])
        v_mix_j = v_mix[:, j] if v_mix.shape[0] else np.array([])
        r_mix_j = r_mix[:, j] if r_mix.shape[0] else np.array([])
        mask_mix_j = mask_mix[:, j] if mask_mix.shape[0] else np.array([])

        z0 = np.array([np.log(max(a[j], 1e-4)), b[j]])

        res = minimize(
            lambda z: item_objective_and_grad_mixed(
                z,
                p_full_j,
                v_full_j,
                r_bin_j,
                p_mix_j,
                v_mix_j,
                r_mix_j,
                mask_mix_j,
                R_b,
                theta_full_p,
                theta_bin,
                theta_mix,
                ridge=ridge,
            )[0],
            z0,
            jac=lambda z: item_objective_and_grad_mixed(
                z,
                p_full_j,
                v_full_j,
                r_bin_j,
                p_mix_j,
                v_mix_j,
                r_mix_j,
                mask_mix_j,
                R_b,
                theta_full_p,
                theta_bin,
                theta_mix,
                ridge=ridge,
            )[1],
            method="L-BFGS-B",
            bounds=[(np.log(0.05), np.log(5.0)), (-5.0, 5.0)],
            options={"maxiter": item_maxiter, "ftol": 1e-7, "gtol": 1e-6, "maxls": 30},
        )

        new_a[j] = np.exp(res.x[0])
        new_b[j] = res.x[1]

    return new_a, new_b


def objective_components_mixed(
    p_full,
    v_full,
    r_bin,
    p_mix,
    v_mix,
    r_mix,
    mask_mix,
    R_b,
    a,
    b,
    theta_full_p,
    theta_bin,
    theta_mix,
    ridge=1e-4,
):
    out = {
        "prob_full": 0.0,
        "bin_full": 0.0,
        "mix_prob": 0.0,
        "mix_bin": 0.0,
        "ridge": 0.0,
    }

    # full probability
    if len(theta_full_p):
        eta = D * a[None, :] * (theta_full_p[:, None] - b[None, :])
        mu = sigmoid(eta)
        valid = np.isfinite(p_full) & np.isfinite(v_full)
        out["prob_full"] = float(
            0.5 * np.sum(((p_full - mu) ** 2 / v_full)[valid])
        )

    # full binary
    if len(theta_bin):
        eta = D * a[None, :] * (theta_bin[:, None] - b[None, :])
        mu = np.clip(sigmoid(eta), 1e-12, 1.0 - 1e-12)
        valid = np.isfinite(r_bin)
        out["bin_full"] = float(
            -np.sum(
                (r_bin * np.log(mu) + (R_b - r_bin) * np.log(1.0 - mu))[valid]
            )
        )

    # mixed
    if len(theta_mix):
        eta = D * a[None, :] * (theta_mix[:, None] - b[None, :])
        mu = sigmoid(eta)
        mu_clip = np.clip(mu, 1e-12, 1.0 - 1e-12)

        prob_idx = (mask_mix == 1) & np.isfinite(p_mix) & np.isfinite(v_mix)
        bin_idx = np.isfinite(r_mix)

        if np.any(prob_idx):
            out["mix_prob"] = float(
                0.5 * np.sum(((p_mix - mu) ** 2 / v_mix)[prob_idx])
            )

        if np.any(bin_idx):
            out["mix_bin"] = float(
                -np.sum(
                    (
                        r_mix * np.log(mu_clip)
                        + (R_b - r_mix) * np.log(1.0 - mu_clip)
                    )[bin_idx]
                )
            )

    if ridge is not None and ridge > 0:
        alpha = np.log(np.clip(a, 1e-12, None))
        out["ridge"] = float(ridge * np.sum(alpha ** 2 + b ** 2))

    out["total"] = (
        out["prob_full"]
        + out["bin_full"]
        + out["mix_prob"]
        + out["mix_bin"]
        + out["ridge"]
    )

    return out


## Approximate conditional standard errors

Information from the two channels is added, including both contributions for shared-theta mixed observations. Item SEs condition on theta; ability SEs condition on items. They do not incorporate joint item/ability uncertainty or uncertainty in estimated s2. Item information is stabilized by 1e-8*I and inverted by pseudoinverse. These are approximate conditional SEs, not the inverse of the full joint Hessian.


In [ ]:
def compute_se_mixed(
    p_full,
    v_full,
    r_bin,
    p_mix,
    v_mix,
    r_mix,
    mask_mix,
    R_b,
    a,
    b,
    theta_full_p,
    theta_bin,
    theta_mix,
):
    n_items = len(a)
    a_se = np.zeros(n_items)
    b_se = np.zeros(n_items)

    for j in range(n_items):
        aj = a[j]
        bj = b[j]
        I = np.zeros((2, 2))

        # full probability group
        if len(theta_full_p):
            valid = np.isfinite(p_full[:, j]) & np.isfinite(v_full[:, j])
            if np.any(valid):
                th = theta_full_p[valid]
                eta = D * aj * (th - bj)
                mu = sigmoid(eta)
                dmu_deta = mu * (1.0 - mu)

                d_eta_da = D * (th - bj)
                d_eta_db = -D * aj * np.ones_like(th)

                G = np.c_[dmu_deta * d_eta_da, dmu_deta * d_eta_db]
                I += G.T @ ((1.0 / v_full[valid, j])[:, None] * G)

        # full binary group
        if len(theta_bin):
            valid = np.isfinite(r_bin[:, j])
            if np.any(valid):
                th = theta_bin[valid]
                eta = D * aj * (th - bj)
                mu = sigmoid(eta)

                G = np.c_[D * (th - bj), -D * aj * np.ones_like(th)]
                I += G.T @ ((R_b * mu * (1.0 - mu))[:, None] * G)

        # mixed group
        if len(theta_mix):
            # probability part
            prob_idx = (
                (mask_mix[:, j] == 1)
                & np.isfinite(p_mix[:, j])
                & np.isfinite(v_mix[:, j])
            )

            if np.any(prob_idx):
                th = theta_mix[prob_idx]
                eta = D * aj * (th - bj)
                mu = sigmoid(eta)
                dmu_deta = mu * (1.0 - mu)

                d_eta_da = D * (th - bj)
                d_eta_db = -D * aj * np.ones_like(th)

                G = np.c_[dmu_deta * d_eta_da, dmu_deta * d_eta_db]
                I += G.T @ ((1.0 / v_mix[prob_idx, j])[:, None] * G)

            # binary part
            bin_idx = np.isfinite(r_mix[:, j])

            if np.any(bin_idx):
                th = theta_mix[bin_idx]
                eta = D * aj * (th - bj)
                mu = sigmoid(eta)

                G = np.c_[D * (th - bj), -D * aj * np.ones_like(th)]
                I += G.T @ ((R_b * mu * (1.0 - mu))[:, None] * G)

        cov = np.linalg.pinv(I + 1e-8 * np.eye(2))
        a_se[j] = np.sqrt(max(cov[0, 0], 1e-12))
        b_se[j] = np.sqrt(max(cov[1, 1], 1e-12))

    # theta SEs
    theta_full_p_se = []
    for g, t in enumerate(theta_full_p):
        eta = D * a * (t - b)
        mu = sigmoid(eta)
        dmu_dtheta = mu * (1.0 - mu) * D * a

        valid = np.isfinite(p_full[g]) & np.isfinite(v_full[g])
        info = np.sum((dmu_dtheta[valid] ** 2) / v_full[g, valid])
        theta_full_p_se.append(1.0 / np.sqrt(max(info, 1e-12)))
    theta_full_p_se = np.array(theta_full_p_se)

    theta_bin_se = []
    for g, t in enumerate(theta_bin):
        eta = D * a * (t - b)
        mu = sigmoid(eta)

        valid = np.isfinite(r_bin[g])
        info = np.sum((D ** 2) * (a[valid] ** 2) * R_b * mu[valid] * (1.0 - mu[valid]))
        theta_bin_se.append(1.0 / np.sqrt(max(info, 1e-12)))
    theta_bin_se = np.array(theta_bin_se)

    theta_mix_se = []
    for g, t in enumerate(theta_mix):
        eta = D * a * (t - b)
        mu = sigmoid(eta)

        info = 0.0

        prob_idx = (
            (mask_mix[g] == 1)
            & np.isfinite(p_mix[g])
            & np.isfinite(v_mix[g])
        )
        if np.any(prob_idx):
            dmu_dtheta = mu[prob_idx] * (1.0 - mu[prob_idx]) * D * a[prob_idx]
            info += np.sum((dmu_dtheta ** 2) / v_mix[g, prob_idx])

        bin_idx = np.isfinite(r_mix[g])
        if np.any(bin_idx):
            info += np.sum(
                (D ** 2)
                * (a[bin_idx] ** 2)
                * R_b
                * mu[bin_idx]
                * (1.0 - mu[bin_idx])
            )

        theta_mix_se.append(1.0 / np.sqrt(max(info, 1e-12)))

    theta_mix_se = np.array(theta_mix_se)

    return a_se, b_se, theta_full_p_se, theta_bin_se, theta_mix_se


## Fitting API

Rows are respondents (LLMs/model conditions), columns are items. Matrices are processed positionally: align item columns across groups, and row indices within each group, before calling. None represents an absent group; use NaN for missing observations. For a mixed group supply p_mix, s2_mix, r_mix and mask_mix with matching shapes; use all-NaN r_mix for probability-only mixed rows. Do not duplicate the same respondent in the full groups if you want a single shared theta: put both channels in the mixed group. The outer loop stops only when BOTH relative objective change <1e-6 and max parameter change <1e-4, or at 1000 iterations. Item limit is also 1000. Check converged and history; hitting a limit is retained as a nonconverged result.


In [ ]:
def fit_hybrid_irt_with_itemwise_mixed(
    p_full=None,
    s2_full=None,
    r_bin=None,
    p_mix=None,
    s2_mix=None,
    r_mix=None,
    mask_mix=None,
    R_l=5,
    R_b=5,
    c_tau=0.25,
    init_ab=None,
    outer_iter=1000,
    item_maxiter=1000,
    tol_obj=1e-6,
    tol_param=1e-4,
    ridge=1e-4,
    verbose=True,
    print_every=1,
):
    """
    Fit IRT with:
      1. fully probability-observed LLMs
      2. fully binary-observed LLMs
      3. item-wise mixed LLMs

    Data layout:
      rows = LLM/model-condition
      columns = items
    """

    n_items = infer_n_items(p_full, s2_full, r_bin, p_mix, s2_mix, r_mix, mask_mix)

    p_full = to_numpy_or_empty(p_full, n_items)
    s2_full = to_numpy_or_empty(s2_full, n_items)
    r_bin = to_numpy_or_empty(r_bin, n_items)

    p_mix = to_numpy_or_empty(p_mix, n_items)
    s2_mix = to_numpy_or_empty(s2_mix, n_items)
    r_mix = to_numpy_or_empty(r_mix, n_items)

    if mask_mix is None:
        mask_mix = np.empty((0, n_items))
    elif isinstance(mask_mix, pd.DataFrame):
        mask_mix = mask_mix.to_numpy(dtype=float)
    else:
        mask_mix = np.asarray(mask_mix, dtype=float)

    # validation
    if p_full.shape != s2_full.shape:
        raise ValueError("p_full and s2_full must have the same shape.")

    if p_mix.shape != s2_mix.shape:
        raise ValueError("p_mix and s2_mix must have the same shape.")

    if p_mix.shape != r_mix.shape or p_mix.shape != mask_mix.shape:
        raise ValueError("p_mix, r_mix, and mask_mix must have the same shape.")


    # Fail early on malformed inputs instead of returning arbitrary or non-finite fits.
    matrices = [p_full, s2_full, r_bin, p_mix, s2_mix, r_mix, mask_mix]
    if any(x.ndim != 2 or x.shape[1] != n_items for x in matrices):
        raise ValueError("All matrices must be 2D with the same item count.")
    if n_items < 1 or sum(x.shape[0] for x in (p_full, r_bin, p_mix)) < 2:
        raise ValueError("At least one item and two fitted respondents are required.")
    if not np.isfinite(R_l) or R_l <= 0:
        raise ValueError("R_l must be positive.")
    if not np.isfinite(R_b) or R_b < 1 or R_b != int(R_b):
        raise ValueError("R_b must be a positive integer.")
    if not np.isfinite(c_tau) or c_tau < 0:
        raise ValueError("c_tau must be finite and nonnegative.")
    if outer_iter < 1 or item_maxiter < 1 or print_every < 1:
        raise ValueError("Iteration limits and print_every must be positive.")
    if tol_obj <= 0 or tol_param <= 0:
        raise ValueError("Convergence tolerances must be positive.")
    if ridge is not None and (not np.isfinite(ridge) or ridge < 0):
        raise ValueError("ridge must be nonnegative or None.")
    if any(np.isinf(x).any() for x in matrices):
        raise ValueError("Use NaN for missing observations, not infinity.")
    if not np.isin(mask_mix, [0, 1]).all():
        raise ValueError("mask_mix must contain only 0 and 1.")
    for counts in (r_bin, r_mix):
        finite = counts[np.isfinite(counts)]
        if np.any((finite < 0) | (finite > R_b) | (finite != np.floor(finite))):
            raise ValueError("Binomial counts must be integers in [0, R_b].")
    for probabilities, variances in ((p_full, s2_full), (p_mix, s2_mix)):
        finite = probabilities[np.isfinite(probabilities)]
        if np.any((finite < 0) | (finite > 1)):
            raise ValueError("Probability observations must lie in [0, 1].")
        if np.any(variances[np.isfinite(variances)] < 0):
            raise ValueError("Sample variances must be nonnegative.")
    full_available = np.isfinite(p_full) & np.isfinite(s2_full)
    mix_available = (mask_mix == 1) & np.isfinite(p_mix) & np.isfinite(s2_mix)
    for available, variances in ((full_available, s2_full), (mix_available, s2_mix)):
        if np.any((variances[available] + c_tau) / R_l <= 0):
            raise ValueError("Every used probability observation needs positive analysis variance.")
    if np.any(np.isfinite(p_full) & ~full_available):
        raise ValueError("Finite p_full observations require finite s2_full.")
    if np.any((mask_mix == 1) & np.isfinite(p_mix) & ~mix_available):
        raise ValueError("Used p_mix observations require finite s2_mix.")
    observed = np.vstack([full_available, np.isfinite(r_bin), mix_available | np.isfinite(r_mix)])
    if not observed.any(axis=0).all() or not observed.any(axis=1).all():
        raise ValueError("Every fitted respondent and item must have an observation.")
    if init_ab is not None:
        if len(init_ab) != 2 or any(np.shape(x) != (n_items,) for x in init_ab):
            raise ValueError("init_ab must contain two vectors of length n_items.")
        if not all(np.isfinite(x).all() for x in init_ab) or np.any(np.asarray(init_ab[0]) <= 0):
            raise ValueError("init_ab must be finite with positive discrimination.")

    v_full = make_variance_from_s2(s2_full, R_l, c_tau=c_tau)
    v_mix = make_variance_from_s2(s2_mix, R_l, c_tau=c_tau)

    a, b, theta_full_p, theta_bin, theta_mix = initialize_params_mixed(
        p_full,
        v_full,
        r_bin,
        p_mix,
        v_mix,
        r_mix,
        mask_mix,
        R_b,
        init_ab=init_ab,
    )

    history = []

    prev_comp = objective_components_mixed(
        p_full,
        v_full,
        r_bin,
        p_mix,
        v_mix,
        r_mix,
        mask_mix,
        R_b,
        a,
        b,
        theta_full_p,
        theta_bin,
        theta_mix,
        ridge=ridge,
    )
    prev_obj = prev_comp["total"]

    prev_a = a.copy()
    prev_b = b.copy()
    prev_theta_full_p = theta_full_p.copy()
    prev_theta_bin = theta_bin.copy()
    prev_theta_mix = theta_mix.copy()

    converged = False
    stop_reason = "max_iter_reached"

    for it in range(1, outer_iter + 1):
        if len(theta_full_p):
            theta_full_p = update_theta_full_probability(p_full, v_full, a, b)

        if len(theta_bin):
            theta_bin = update_theta_binary(r_bin, R_b, a, b)

        if len(theta_mix):
            theta_mix = update_theta_mixed(
                p_mix,
                v_mix,
                r_mix,
                mask_mix,
                R_b,
                a,
                b,
            )

        a, b = update_items_mixed(
            p_full,
            v_full,
            r_bin,
            p_mix,
            v_mix,
            r_mix,
            mask_mix,
            R_b,
            a,
            b,
            theta_full_p,
            theta_bin,
            theta_mix,
            item_maxiter=item_maxiter,
            ridge=ridge,
        )

        a, b, theta_full_p, theta_bin, theta_mix = standardize_params_mixed(
            a,
            b,
            theta_full_p,
            theta_bin,
            theta_mix,
        )

        comp = objective_components_mixed(
            p_full,
            v_full,
            r_bin,
            p_mix,
            v_mix,
            r_mix,
            mask_mix,
            R_b,
            a,
            b,
            theta_full_p,
            theta_bin,
            theta_mix,
            ridge=ridge,
        )

        obj = comp["total"]
        abs_obj_change = abs(prev_obj - obj)
        rel_obj_change = abs_obj_change / (abs(prev_obj) + 1e-12)

        max_delta_a = np.max(np.abs(a - prev_a)) if len(a) else 0.0
        max_delta_b = np.max(np.abs(b - prev_b)) if len(b) else 0.0
        max_delta_theta_full_p = (
            np.max(np.abs(theta_full_p - prev_theta_full_p))
            if len(theta_full_p)
            else 0.0
        )
        max_delta_theta_bin = (
            np.max(np.abs(theta_bin - prev_theta_bin))
            if len(theta_bin)
            else 0.0
        )
        max_delta_theta_mix = (
            np.max(np.abs(theta_mix - prev_theta_mix))
            if len(theta_mix)
            else 0.0
        )

        max_param_change = max(
            max_delta_a,
            max_delta_b,
            max_delta_theta_full_p,
            max_delta_theta_bin,
            max_delta_theta_mix,
        )

        meets_obj = rel_obj_change < tol_obj
        meets_param = max_param_change < tol_param
        converged_now = meets_obj and meets_param

        row = {
            "iteration": it,
            "objective": obj,
            "objective_prob_full": comp["prob_full"],
            "objective_bin_full": comp["bin_full"],
            "objective_mix_prob": comp["mix_prob"],
            "objective_mix_bin": comp["mix_bin"],
            "objective_ridge": comp["ridge"],
            "relative_objective_change": rel_obj_change,
            "max_parameter_change": max_param_change,
            "max_delta_a": max_delta_a,
            "max_delta_b": max_delta_b,
            "max_delta_theta_full_p": max_delta_theta_full_p,
            "max_delta_theta_bin": max_delta_theta_bin,
            "max_delta_theta_mix": max_delta_theta_mix,
            "meets_obj_tol": meets_obj,
            "meets_param_tol": meets_param,
            "converged": converged_now,
        }
        history.append(row)

        if verbose and (it == 1 or it % print_every == 0 or converged_now):
            print(
                f"[iter {it:03d}] "
                f"obj={obj:.6f} | "
                f"rel_dL={rel_obj_change:.3e} | "
                f"max_dparam={max_param_change:.3e} | "
                f"converged={converged_now}"
            )

        if converged_now:
            converged = True
            stop_reason = "converged"
            break

        prev_obj = obj
        prev_a = a.copy()
        prev_b = b.copy()
        prev_theta_full_p = theta_full_p.copy()
        prev_theta_bin = theta_bin.copy()
        prev_theta_mix = theta_mix.copy()

    a_se, b_se, theta_full_p_se, theta_bin_se, theta_mix_se = compute_se_mixed(
        p_full,
        v_full,
        r_bin,
        p_mix,
        v_mix,
        r_mix,
        mask_mix,
        R_b,
        a,
        b,
        theta_full_p,
        theta_bin,
        theta_mix,
    )

    history_df = pd.DataFrame(history)

    return {
        "a": a,
        "b": b,
        "theta_full_probability": theta_full_p,
        "theta_binary": theta_bin,
        "theta_mixed": theta_mix,
        "a_se": a_se,
        "b_se": b_se,
        "theta_full_probability_se": theta_full_p_se,
        "theta_binary_se": theta_bin_se,
        "theta_mixed_se": theta_mix_se,
        "history": history_df,
        "converged": converged,
        "stop_reason": stop_reason,
        "n_iterations": len(history_df),
        "final_objective": history_df["objective"].iloc[-1] if len(history_df) else np.nan,
        "final_relative_objective_change": (
            history_df["relative_objective_change"].iloc[-1]
            if len(history_df)
            else np.nan
        ),
        "final_max_parameter_change": (
            history_df["max_parameter_change"].iloc[-1]
            if len(history_df)
            else np.nan
        ),
    }


## Simulation data generator

The random-number generator and draw order are preserved. See the full simulation specification above. All generated groups are retained even when a particular fit omits some groups: removing those draws would change later observations for the same seed. Changing sample sizes does not create nested samples. At fixed sizes and seed, changing only binary_fraction changes only the mask, using common random numbers.


In [ ]:
@dataclass
class MixedSimulationData:
    p_full_df: pd.DataFrame
    s2_full_df: pd.DataFrame
    r_bin_df: pd.DataFrame
    p_mix_df: pd.DataFrame
    s2_mix_df: pd.DataFrame
    r_mix_df: pd.DataFrame
    mask_mix_df: pd.DataFrame

    # true parameters for simulation diagnostics
    a_true: np.ndarray
    b_true: np.ndarray
    theta_full_true: np.ndarray
    theta_bin_true: np.ndarray
    theta_mix_true: np.ndarray

    # optional underlying means
    mu_full: np.ndarray
    mu_bin: np.ndarray
    mu_mix: np.ndarray

    R_l: int
    R_b: int
    binary_fraction: float


def simulate_itemwise_mixed_dataframes(
    n_full_probability=100,
    n_full_binary=100,
    n_mixed=100,
    n_items=100,
    R_l=5,
    R_b=5,
    binary_fraction=0.3,
    seed=42,
    item_b_low=-2.0,
    item_b_high=2.0,
    a_low=0.5,
    a_high=2.0,
):
    """
    Generate simulation DataFrames for item-wise mixed hybrid IRT.

    Generated DataFrames:
      1. p_full_df   : mean correct probabilities for fully logprobs-measured LLMs
      2. s2_full_df  : variance of repeated correct probabilities for fully logprobs-measured LLMs
      3. r_bin_df    : correct counts for fully binary-measured LLMs
      4. p_mix_df    : mean correct probabilities for item-wise mixed LLMs
      5. s2_mix_df   : variance of repeated correct probabilities for item-wise mixed LLMs
      6. r_mix_df    : correct counts for item-wise mixed LLMs
      7. mask_mix_df : 1 = enable probability IN ADDITION to binomial, 0 = binomial only

    Notes:
      - p_mix_df, s2_mix_df, and r_mix_df are generated mechanically for all cells.
      - mask_mix_df gates probability only; all finite r_mix counts are used.
      - binary_fraction controls the fraction of 0s in mask_mix_df.
    """
    if not (0.0 <= binary_fraction <= 1.0):
        raise ValueError("binary_fraction must be between 0 and 1.")


    sizes = (n_full_probability, n_full_binary, n_mixed, n_items)
    if any(not isinstance(x, (int, np.integer)) or x < 0 for x in sizes):
        raise ValueError("Group sizes and item count must be nonnegative integers.")
    if n_items < 1 or sum(sizes[:3]) < 2:
        raise ValueError("Generate at least one item and two respondents.")
    if not isinstance(R_l, (int, np.integer)) or R_l < 2:
        raise ValueError("R_l must be an integer >= 2 to compute sample variance (ddof=1).")
    if not isinstance(R_b, (int, np.integer)) or R_b < 1:
        raise ValueError("R_b must be a positive integer.")
    if not (0 < a_low < a_high and item_b_low < item_b_high):
        raise ValueError("Require 0 < a_low < a_high and item_b_low < item_b_high.")

    rng = np.random.default_rng(seed)

    # Item parameters
    a_true = rng.uniform(a_low, a_high, n_items)
    b_true = rng.uniform(item_b_low, item_b_high, n_items)

    # Ability parameters
    theta_full_true = rng.normal(0.0, 1.0, n_full_probability)
    theta_bin_true = rng.normal(0.0, 1.0, n_full_binary)
    theta_mix_true = rng.normal(0.0, 1.0, n_mixed)

    item_names = [f"item_{j+1:03d}" for j in range(n_items)]
    full_names = [f"prob_llm_{g+1:03d}" for g in range(n_full_probability)]
    bin_names = [f"bin_llm_{g+1:03d}" for g in range(n_full_binary)]
    mix_names = [f"mix_llm_{g+1:03d}" for g in range(n_mixed)]

    def generate_probability_observations(theta):
        """
        Generate repeated probability-scale observations.

        p_rep = mu + Normal(0, sigma_p^2)
        clipped to [0, 1].

        sigma_p is largest around mu=0.5 and smallest near 0/1:
        sigma_p = 0.01 + 0.2 * np.sqrt(mu * (1.0 - mu))
        """
        eta = D * a_true[None, :] * (theta[:, None] - b_true[None, :])
        mu = sigmoid(eta)

        sigma_p = 0.01 + 0.2 * np.sqrt(mu * (1.0 - mu))
        sigma_p = np.maximum(sigma_p, 0.0)

        p_rep = mu[:, :, None] + rng.normal(
            loc=0.0,
            scale=sigma_p[:, :, None],
            size=(len(theta), n_items, R_l),
        )

        p_rep = np.clip(p_rep, 0.0, 1.0)

        p_bar = p_rep.mean(axis=2)
        s2_p = p_rep.var(axis=2, ddof=1)

        return p_bar, s2_p, mu

    def generate_binary_counts(theta):
        """
        Generate Binomial correct counts.
        """
        eta = D * a_true[None, :] * (theta[:, None] - b_true[None, :])
        mu = sigmoid(eta)
        r = rng.binomial(R_b, mu)
        return r, mu

    # 1-2. full probability group
    p_full, s2_full, mu_full = generate_probability_observations(theta_full_true)

    # 3. full binary group
    r_bin, mu_bin = generate_binary_counts(theta_bin_true)

    # 4-6. item-wise mixed group
    p_mix, s2_mix, mu_mix = generate_probability_observations(theta_mix_true)
    r_mix, _ = generate_binary_counts(theta_mix_true)

    # 7. mask for mixed group
    # 1 = probability + binomial; 0 = binomial only (r_mix is finite here).
    mask_mix = (rng.random(size=(n_mixed, n_items)) >= binary_fraction).astype(int)

    # Convert to DataFrames
    p_full_df = pd.DataFrame(p_full, index=full_names, columns=item_names)
    s2_full_df = pd.DataFrame(s2_full, index=full_names, columns=item_names)

    r_bin_df = pd.DataFrame(r_bin, index=bin_names, columns=item_names)

    p_mix_df = pd.DataFrame(p_mix, index=mix_names, columns=item_names)
    s2_mix_df = pd.DataFrame(s2_mix, index=mix_names, columns=item_names)
    r_mix_df = pd.DataFrame(r_mix, index=mix_names, columns=item_names)
    mask_mix_df = pd.DataFrame(mask_mix, index=mix_names, columns=item_names)

    return MixedSimulationData(
        p_full_df=p_full_df,
        s2_full_df=s2_full_df,
        r_bin_df=r_bin_df,
        p_mix_df=p_mix_df,
        s2_mix_df=s2_mix_df,
        r_mix_df=r_mix_df,
        mask_mix_df=mask_mix_df,
        a_true=a_true,
        b_true=b_true,
        theta_full_true=theta_full_true,
        theta_bin_true=theta_bin_true,
        theta_mix_true=theta_mix_true,
        mu_full=mu_full,
        mu_bin=mu_bin,
        mu_mix=mu_mix,
        R_l=R_l,
        R_b=R_b,
        binary_fraction=binary_fraction,
    )


## Selecting fitted groups

These four scenarios share the same generated data. Omitting a group from fitting does not delete its random draws. The mixed scenario includes probability + binomial measurements for each mask-1 mixed cell, using one theta per row.


In [ ]:
SCENARIOS = {
    "probability_only": (True, False, False),
    "binary_only": (False, True, False),
    "no_mixed": (True, True, False),
    "mixed": (True, True, True),
}

FIT_DEFAULTS = {
    "c_tau": 0.25, "outer_iter": 1000, "item_maxiter": 1000,
    "tol_obj": 1e-6, "tol_param": 1e-4, "ridge": 1e-4,
    "verbose": False, "print_every": 1,
}


def fit_simulation(sim, scenario="mixed", **fit_options):
    """Fit a selected subset of generated groups without altering simulated data."""
    if scenario not in SCENARIOS:
        raise ValueError(f"scenario must be one of {tuple(SCENARIOS)}")
    use_p, use_b, use_m = SCENARIOS[scenario]
    options = dict(FIT_DEFAULTS)
    options.update(fit_options)
    return fit_hybrid_irt_with_itemwise_mixed(
        p_full=sim.p_full_df if use_p else None,
        s2_full=sim.s2_full_df if use_p else None,
        r_bin=sim.r_bin_df if use_b else None,
        p_mix=sim.p_mix_df if use_m else None,
        s2_mix=sim.s2_mix_df if use_m else None,
        r_mix=sim.r_mix_df if use_m else None,
        mask_mix=sim.mask_mix_df if use_m else None,
        R_l=sim.R_l, R_b=sim.R_b, **options,
    )


def fitted_truth(fit, sim, aligned=True):
    """Align true parameters using ONLY the groups included in this fit."""
    truth = [sim.a_true.copy(), sim.b_true.copy()]
    for key, values in (
        ("theta_full_probability", sim.theta_full_true),
        ("theta_binary", sim.theta_bin_true),
        ("theta_mixed", sim.theta_mix_true),
    ):
        n = len(fit[key])
        if n not in (0, len(values)):
            raise ValueError("Fit and simulation group sizes do not match.")
        truth.append(values.copy() if n else np.array([]))
    return standardize_params_mixed(*truth) if aligned else tuple(truth)


def parameter_tables(fit, sim):
    """Return separate item and respondent tables; row counts need not match."""
    raw = fitted_truth(fit, sim, aligned=False)
    aligned = fitted_truth(fit, sim, aligned=True)
    items = pd.DataFrame({"item": sim.p_full_df.columns})
    for i, name in enumerate(("a", "b")):
        items[f"{name}_true_raw"] = raw[i]
        items[f"{name}_true_aligned"] = aligned[i]
        items[f"{name}_estimate"] = fit[name]
        items[f"{name}_se"] = fit[f"{name}_se"]
    frames = []
    for i, (key, index) in enumerate((
        ("theta_full_probability", sim.p_full_df.index),
        ("theta_binary", sim.r_bin_df.index),
        ("theta_mixed", sim.p_mix_df.index),
    ), start=2):
        if len(fit[key]):
            frames.append(pd.DataFrame({
                "respondent": index, "group": key,
                "theta_true_raw": raw[i], "theta_true_aligned": aligned[i],
                "theta_estimate": fit[key], "theta_se": fit[f"{key}_se"],
            }))
    return items, pd.concat(frames, ignore_index=True)


def summarize_result(fit, sim):
    """Report aligned recovery, raw-scale recovery, conditional RMSSE, and convergence."""
    aligned = fitted_truth(fit, sim, aligned=True)
    raw = fitted_truth(fit, sim, aligned=False)
    mask = sim.mask_mix_df.to_numpy()
    has_mixed = len(fit["theta_mixed"]) > 0
    out = {
        "binary_fraction_target": sim.binary_fraction,
        "binary_fraction_actual": float((mask == 0).mean()) if has_mixed and mask.size else np.nan,
        "converged": fit["converged"],
        "stop_reason": fit["stop_reason"],
        "n_iterations": fit["n_iterations"],
        "final_rel_obj_change": fit["final_relative_objective_change"],
        "final_max_param_change": fit["final_max_parameter_change"],
    }
    for i, key in enumerate(("a", "b", "theta_full_probability", "theta_binary", "theta_mixed")):
        estimate, se = fit[key], fit[f"{key}_se"]
        out[f"rmse_{key}"] = rmse(estimate, aligned[i])
        out[f"rmse_{key}_raw"] = rmse(estimate, raw[i])
        out[f"rmsse_{key}"] = float(np.sqrt(np.mean(se**2))) if len(se) else np.nan
        if key in ("a", "b"):
            out[f"corr_{key}"] = corr(estimate, aligned[i])
            out[f"median_{key}_se"] = float(np.median(se))
            out[f"p90_{key}_se"] = float(np.quantile(se, 0.9))
    for key in ("prob_full", "bin_full", "mix_prob", "mix_bin"):
        out[f"objective_{key}"] = float(fit["history"][f"objective_{key}"].iloc[-1])
    return out


## Replicate summaries and one reusable simulation runner

RMSSE means sqrt(mean(SE**2)) within a fit, not arithmetic mean SE. Across seeds, report the mean of each metric, sample SD, Monte Carlo SE = SD/sqrt(number of finite replicates), and empirical 2.5%/97.5% quantiles. Quantiles describe the replicate distribution, not a confidence interval for its mean. Nonconverged fits are retained, with their convergence rate; inspect them before interpretation.


In [ ]:
def summarize_replicates(raw):
    """Aggregate by experiment, scenario, varied parameter and its value."""
    if raw.empty:
        raise ValueError("No simulation results to summarize.")
    grouping = ["experiment", "scenario", "vary", "value"]
    metrics = [c for c in raw.columns if c.startswith(("rmse_", "rmsse_", "corr_", "median_", "p90_", "final_"))]
    metrics += ["binary_fraction_actual", "n_iterations"]
    rows = []
    for keys, group in raw.groupby(grouping, sort=False, dropna=False):
        row = dict(zip(grouping, keys))
        row.update(n_rep=len(group), n_seed=group["seed"].nunique(),
                   convergence_rate=float(group["converged"].mean()))
        for metric in metrics:
            values = group[metric].dropna()
            n = len(values)
            sd = float(values.std(ddof=1)) if n > 1 else np.nan
            row.update({
                f"{metric}_mean": float(values.mean()) if n else np.nan,
                f"{metric}_sd": sd,
                f"{metric}_se": sd / np.sqrt(n) if n > 1 else np.nan,
                f"{metric}_n": n,
                f"{metric}_q025": float(values.quantile(.025)) if n else np.nan,
                f"{metric}_q975": float(values.quantile(.975)) if n else np.nan,
            })
        rows.append(row)
    return pd.DataFrame(rows)


def run_simulation_grid(experiment, *, values=None, seeds=None, fit_options=None,
                        output_dir=None, progress=True):
    """Run a named experiment. Overrides make small verification runs possible.

    Full defaults live in EXPERIMENTS. Each value/seed generates one dataset,
    reused across its scenarios. Optional CSV output is checkpointed after each fit.
    Existing files with the same experiment name are replaced by this run.
    """
    if experiment not in EXPERIMENTS:
        raise ValueError(f"Unknown experiment: {experiment}")
    spec = EXPERIMENTS[experiment]
    values = list(spec["values"] if values is None else values)
    seeds = list(spec["seeds"] if seeds is None else seeds)
    if not values or not seeds or len(set(values)) != len(values) or len(set(seeds)) != len(seeds):
        raise ValueError("Supply nonempty, unique values and seeds.")
    rows = []
    destination = Path(output_dir) if output_dir is not None else None
    if destination is not None:
        destination.mkdir(parents=True, exist_ok=True)
    for value in values:
        for seed in seeds:
            generation = dict(spec["generation"])
            if spec["vary"] != "none":
                generation[spec["vary"]] = value
            sim = simulate_itemwise_mixed_dataframes(seed=seed, **generation)
            for scenario in spec["scenarios"]:
                start = time.perf_counter()
                fit = fit_simulation(sim, scenario, **(fit_options or {}))
                row = summarize_result(fit, sim)
                row.update(experiment=experiment, scenario=scenario, vary=spec["vary"],
                           value=value, seed=seed, elapsed_seconds=time.perf_counter()-start)
                row.update({f"generated_{k}": v for k, v in generation.items()})
                for key, fit_key in (("probability", "theta_full_probability"),
                                     ("binary", "theta_binary"), ("mixed", "theta_mixed")):
                    row[f"n_fitted_{key}"] = len(fit[fit_key])
                row["c_tau"] = (fit_options or {}).get("c_tau", 0.25)
                rows.append(row)
                if destination is not None:
                    pd.DataFrame(rows).to_csv(destination/f"{experiment}_raw.csv", index=False)
                if progress:
                    print(f"{experiment}: {spec['vary']}={value}, seed={seed}, "
                          f"{scenario}, converged={fit['converged']}, iterations={fit['n_iterations']}")
    raw = pd.DataFrame(rows)
    summary = summarize_replicates(raw)
    if destination is not None:
        summary.to_csv(destination/f"{experiment}_summary.csv", index=False)
        manifest = dict(spec, values=values, seeds=seeds,
                        fit_options={**FIT_DEFAULTS, **(fit_options or {})})
        (destination/f"{experiment}_settings.json").write_text(
            json.dumps(manifest, indent=2), encoding="utf-8")
    return raw, summary


## Simulation conditions and source correspondence

Full grids preserve the original values, seeds, generated group sizes and draw order. Generated groups omitted from fitting are explicitly distinguished from fitted sample sizes. The four sample-size grids correspond to slide 12 Experiment 2. Item-count and mixed-fraction grids are supplementary. The mixed item-count experiment now correctly fits the mixed group; the source accidentally invoked its no-mixed runner.


In [ ]:
SAMPLE_SIZES = list(range(10, 101, 10)) + list(range(200, 1001, 100))
ITEM_COUNTS = list(range(10, 101, 10)) + list(range(150, 501, 50))
EARLY_ITEM_COUNTS = list(range(10, 101, 10)) + [125, 150, 175, 200, 250, 300, 350, 400, 450, 500]
FRACTIONS = [i/20 for i in range(1, 20)]
TEN_SEEDS = list(range(42, 52))


def experiment_spec(vary, values, scenarios, *, seeds=TEN_SEEDS, n_p=50, n_b=50,
                    n_m=50, fraction=.3, source_cells=()):
    """Build a condition record; source cell numbers are 1-based, including Markdown."""
    return {
        "vary": vary, "values": list(values), "seeds": list(seeds),
        "scenarios": list(scenarios), "source_cells": list(source_cells),
        "generation": {"n_full_probability": n_p, "n_full_binary": n_b, "n_mixed": n_m,
                       "n_items": 100, "R_l": 5, "R_b": 5, "binary_fraction": fraction,
                       "a_low": .5, "a_high": 2.0, "item_b_low": -2.0, "item_b_high": 2.0},
    }


EXPERIMENTS = {
    "recovery": experiment_spec("none", [0], SCENARIOS, seeds=[42], n_p=100, n_b=100,
                                source_cells=[19, 21, 23, 25]),
    "sample_binary": experiment_spec("n_full_binary", SAMPLE_SIZES, ["binary_only"], source_cells=[49, 50]),
    "sample_probability": experiment_spec("n_full_probability", SAMPLE_SIZES, ["probability_only"], source_cells=[41, 42]),
    "sample_binary_plus_probability50": experiment_spec("n_full_binary", SAMPLE_SIZES, ["no_mixed"], source_cells=[66, 67]),
    "sample_probability_plus_binary50": experiment_spec("n_full_probability", SAMPLE_SIZES, ["no_mixed"], source_cells=[58, 59]),
    "items_binary": experiment_spec("n_items", ITEM_COUNTS, ["binary_only"], n_p=100, n_b=100, source_cells=[80, 81]),
    "items_probability": experiment_spec("n_items", ITEM_COUNTS, ["probability_only"], n_p=100, n_b=100, source_cells=[84, 85]),
    "items_no_mixed": experiment_spec("n_items", ITEM_COUNTS, ["no_mixed"], n_p=100, n_b=100, source_cells=[87, 88]),
    "items_mixed": experiment_spec("n_items", ITEM_COUNTS, ["mixed"], n_p=100, n_b=100, fraction=.5, source_cells=[90, 91]),
    "mixed_fraction": experiment_spec("binary_fraction", FRACTIONS, ["mixed"], source_cells=[74, 75]),
    "mixed_fraction_50_replicates": experiment_spec("binary_fraction", [.25, .5, .75], ["mixed"],
                    seeds=range(1, 51), n_p=100, n_b=100, n_m=100, source_cells=[93, 94]),
    "early_items_probability50": experiment_spec("n_items", EARLY_ITEM_COUNTS, ["probability_only"], seeds=[42], source_cells=[31]),
    "compare_groups50": experiment_spec("binary_fraction", [i/10 for i in range(1,10)], SCENARIOS, seeds=[42], source_cells=[28, 29, 34, 38]),
    "compare_probability100_binary50": experiment_spec("none", [0], SCENARIOS, seeds=[42], n_p=100, source_cells=[35]),
    "compare_groups100_fraction50": experiment_spec("none", [0], SCENARIOS, seeds=[42], n_p=100, n_b=100, fraction=.5, source_cells=[36]),
    "initial_mixed_demo": experiment_spec("none", [0], ["mixed"], seeds=[42], n_p=100, n_b=100, n_m=100, source_cells=[13, 15]),
}


def experiment_catalog():
    """Show generated and fitted sizes without running any simulations."""
    rows = []
    for name, spec in EXPERIMENTS.items():
        for scenario in spec["scenarios"]:
            use = SCENARIOS[scenario]
            g = spec["generation"]
            row = {"experiment": name, "scenario": scenario, "vary": spec["vary"],
                   "values": spec["values"], "seeds": spec["seeds"],
                   "fits": len(spec["values"])*len(spec["seeds"])}
            for key, included in zip(("n_full_probability", "n_full_binary", "n_mixed"), use):
                generated = "varied" if spec["vary"] == key else g[key]
                row[f"generated_{key}"] = generated
                row[f"fitted_{key}"] = generated if included else 0
            rows.append(row)
    return pd.DataFrame(rows)


## Plotting helpers

Recovery plots explicitly select raw or aligned truth. SE plots display mean within-fit RMSSE across seeds, with Monte Carlo SE error bars. No curve is fitted automatically. Matplotlib is only needed when plotting.


In [ ]:
def plot_recovery(fit, sim, *, truth_scale="aligned"):
    """Plot item/ability recovery with approximate conditional SE bars."""
    import matplotlib.pyplot as plt
    if truth_scale not in ("raw", "aligned"):
        raise ValueError("truth_scale must be 'raw' or 'aligned'.")
    truth = fitted_truth(fit, sim, aligned=truth_scale == "aligned")
    keys = ["a", "b", "theta_full_probability", "theta_binary", "theta_mixed"]
    present = [(i,k) for i,k in enumerate(keys) if len(fit[k])]
    fig, axes = plt.subplots(1, len(present), figsize=(4*len(present), 3.6), squeeze=False)
    for ax, (i, key) in zip(axes[0], present):
        x, y = truth[i], fit[key]
        ax.errorbar(x, y, yerr=fit[f"{key}_se"], fmt='.', alpha=.65, capsize=1)
        limits = [min(x.min(), y.min()), max(x.max(), y.max())]
        ax.plot(limits, limits, color='black', linewidth=.8)
        ax.set(title=f"{key}\nRMSE={rmse(y,x):.3f}", xlabel=f"True ({truth_scale})", ylabel="Estimate")
    fig.tight_layout()
    return fig


def plot_rmsse(summary, metric="rmsse_a"):
    """Plot seed-mean RMSSE against the varied condition value."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    for (experiment, scenario), group in summary.groupby(["experiment", "scenario"], sort=False):
        group = group.sort_values("value")
        error = group[f"{metric}_se"].to_numpy()
        ax.errorbar(group["value"], group[f"{metric}_mean"],
                    yerr=error if np.isfinite(error).any() else None,
                    fmt='o-', capsize=2, label=f"{experiment}: {scenario}")
    ax.set(xlabel="Varied condition value", ylabel=f"{metric} (mean across seeds)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


## Execution settings

The recovery example uses the original generated sizes (100 probability, 100 binomial, 50 mixed; 100 items; seed 42). It fits four scenarios. The first three correspond to the presentation; the fourth preserves the supplementary mixed group. Large grids stay off by default. Each full sample-size family has 190 fits; the four families together have 760.


In [ ]:
RESULTS_DIR = Path("simulation_results")
RUN_SAMPLE_SIZE_GRIDS = False
RUN_ITEM_COUNT_GRIDS = False
RUN_MIXED_FRACTION_GRIDS = False
RUN_EARLY_EXPLORATORY_GRIDS = False

experiment_catalog()


## Recovery example: four scenarios on identical generated data

This example keeps a separate table for item parameters and for respondents, so unequal counts are supported. Both raw and aligned true values are exported; RMSE naming is explicit. Existing files in RESULTS_DIR with these names are replaced when this cell is rerun. Inspect convergence rather than assuming it.


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sim_recovery = simulate_itemwise_mixed_dataframes(
    **EXPERIMENTS["recovery"]["generation"], seed=42
)
recovery_fits = {}
recovery_rows = []
for scenario in SCENARIOS:
    result = fit_simulation(sim_recovery, scenario)
    recovery_fits[scenario] = result
    recovery_rows.append({"scenario": scenario, **summarize_result(result, sim_recovery)})
    item_table, respondent_table = parameter_tables(result, sim_recovery)
    item_table.to_csv(RESULTS_DIR / f"recovery_{scenario}_items.csv", index=False)
    respondent_table.to_csv(RESULTS_DIR / f"recovery_{scenario}_respondents.csv", index=False)
    result["history"].to_csv(RESULTS_DIR / f"recovery_{scenario}_history.csv", index=False)
recovery_summary = pd.DataFrame(recovery_rows)
recovery_summary.to_csv(RESULTS_DIR / "recovery_summary.csv", index=False)
recovery_summary[["scenario", "rmse_a", "rmse_a_raw", "rmse_b", "rmse_b_raw",
                  "rmsse_a", "rmsse_b", "converged", "n_iterations"]]


## Recovery plots

Choose truth_scale="raw" to inspect the original generating scale, or "aligned" for identified-scale recovery. Error bars are approximate conditional SEs. The mixed plot includes theta_mixed.


In [ ]:
import matplotlib.pyplot as plt

for scenario, result in recovery_fits.items():
    fig = plot_recovery(result, sim_recovery, truth_scale="aligned")
    fig.suptitle(scenario, y=1.05)
    plt.show()


## Explicit demonstration: the same theta uses both probability and binomial

All respondents in this example are in the mixed group, and binary_fraction=0 enables both channels at every item. There are 40 respondents and 30 items; this compact illustration is additional to the source experiments. The assertion checks that the final objective includes both channels and that each respondent has only one estimated theta.


In [ ]:
sim_shared = simulate_itemwise_mixed_dataframes(
    n_full_probability=0, n_full_binary=0, n_mixed=40, n_items=30,
    R_l=5, R_b=5, binary_fraction=0.0, seed=42,
)
fit_shared = fit_simulation(sim_shared, "mixed")
assert len(fit_shared["theta_mixed"]) == 40
assert len(fit_shared["theta_full_probability"]) == len(fit_shared["theta_binary"]) == 0
last = fit_shared["history"].iloc[-1]
assert last["objective_mix_prob"] > 0 and last["objective_mix_bin"] > 0
pd.DataFrame([summarize_result(fit_shared, sim_shared)])


## Full presentation sample-size experiments

Set RUN_SAMPLE_SIZE_GRIDS=True above, then rerun this cell. Original sizes and all ten seeds are used. Raw results are checkpointed after each fit, with condition settings saved as JSON. To try a small subset instead, call run_simulation_grid("sample_binary", values=[20,50], seeds=[42,43]). This does not change the saved full defaults.


In [ ]:
sample_size_results = {}
if RUN_SAMPLE_SIZE_GRIDS:
    for name in ["sample_binary", "sample_probability", "sample_binary_plus_probability50",
                 "sample_probability_plus_binary50"]:
        sample_size_results[name] = run_simulation_grid(name, output_dir=RESULTS_DIR)
    sample_summary = pd.concat([pair[1] for pair in sample_size_results.values()], ignore_index=True)
    for metric in ["rmsse_a", "rmsse_b"]:
        plot_rmsse(sample_summary, metric)
        plt.show()
else:
    print("Full sample-size grids are disabled. Enable RUN_SAMPLE_SIZE_GRIDS to run them.")


## Supplementary item-count experiments

Each original item-count experiment uses 100 probability respondents, 100 binomial respondents and 50 mixed respondents in generation, with only the selected groups fitted. Item counts are 10,20,...,100,150,200,...,500; ten seeds each. Mixed uses binary_fraction=0.5, others retain 0.3. The mixed experiment now calls the mixed scenario; the original call mistakenly used no_mixed.


In [ ]:
item_count_results = {}
if RUN_ITEM_COUNT_GRIDS:
    for name in ["items_binary", "items_probability", "items_no_mixed", "items_mixed"]:
        item_count_results[name] = run_simulation_grid(name, output_dir=RESULTS_DIR)
else:
    print("Item-count grids are disabled. Enable RUN_ITEM_COUNT_GRIDS to run them.")


## Supplementary mixed-fraction experiments

mixed_fraction: 50 respondents per group, fractions 0.05–0.95, ten seeds. mixed_fraction_50_replicates: 100 respondents per group, fractions 0.25/0.50/0.75, seeds 1–50. Binomial is present at every mixed item in both experiments, including where probability is enabled. With the same seed and sizes, probability/count data remain identical across fractions; only the mask changes.


In [ ]:
mixed_fraction_results = {}
if RUN_MIXED_FRACTION_GRIDS:
    for name in ["mixed_fraction", "mixed_fraction_50_replicates"]:
        mixed_fraction_results[name] = run_simulation_grid(name, output_dir=RESULTS_DIR)
else:
    print("Mixed-fraction grids are disabled. Enable RUN_MIXED_FRACTION_GRIDS to run them.")


## Earlier exploratory settings retained as named experiments

These conditions preserve the early item-count sweep, alternative group sizes and mixed demonstrations. Repeated one-seed sample-size cells are represented by the corresponding full grid with seeds=[42]. Their fitting settings are the same; the summary alignment correction still applies.


In [ ]:
exploratory_results = {}
if RUN_EARLY_EXPLORATORY_GRIDS:
    for name in ["early_items_probability50", "compare_groups50",
                 "compare_probability100_binary50", "compare_groups100_fraction50", "initial_mixed_demo"]:
        exploratory_results[name] = run_simulation_grid(name, output_dir=RESULTS_DIR)
else:
    print("Earlier exploratory grids are disabled. Enable RUN_EARLY_EXPLORATORY_GRIDS to run them.")
